# Chain

- 여러 컴포넌트들을 논리적 순서대로 연결하여 복잡한 작업을 수행하는 구조로 복잡한 AI 작업을 체계적이고 효율적으로 구현할 수 있게 해준다.
- 기본 개념
    - 일련의 작업을 구성하는 여러 개별 컴포넌트들을 정의된 순서대로 실행시킨다.
    - 단일 API 호출을 넘어 여러 호출을 논리적 순서로 연결 가능하다.
    - 복잡한 작업을 작은 단계로 분해하여 순차적으로 처리할 수 있다.

- Langchain은 `off-the-shelf chains` 방식과 `LCEL(Langchain Expression Language)`  두가지 방식이 있다.
  - off-the-shelf chains 방식
    - 미리정의된 Chain 클래스를 사용해 체인을 구성하는 방식
    - Langchain의 초기 방식으로 대부분의 class들이 deprecated 되었다.
  - LECL 방식
    - 표현식을 이용해 체인을 구성하는 방식이다.
    - 현재 LangChain은 LCEL(LangChain Expression Language)을 중심으로 발전하고 있다


# Off-the-shelf chains 예제

In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain import PromptTemplate
from langchain_openai import ChatOpenAI
from pprint import pprint

prompt_template = PromptTemplate(
    template='{item}에 어울리는 이름 {count}개를 만들어주세요'
)

model = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=1.0
)

# 1. prompt 생성 -> llm 요청
prompt = prompt_template.invoke({'item':'스마트폰', 'count':5})
result = model.invoke(prompt)
print(result)

content='물론입니다! 스마트폰에 어울리는 멋진 이름 5개를 제안해 드릴게요.\n\n1. **시너지폰 (Synergy Phone)** - 다양한 기능들이 조화롭게 작동하는 스마트폰.\n2. **에버론 (Everon)** - 항상 연결되어 있는 느낌을 주는 이름.\n3. **퓨처엣지 (FutureEdge)** - 미래적인 기술과 디자인을 강조한 이름.\n4. **노바컴 (NovaCom)** - 새로운 커뮤니케이션을 의미하는 혁신적인 이름.\n5. **스마트링크 (SmartLink)** - 사용자와 세상을 연결하는 스마트폰.\n\n이 이름들이 도움이 되길 바랍니다!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 147, 'prompt_tokens': 21, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0705bf87c0', 'finish_reason': 'stop', 'logprobs': None} id='run-bfb09f22-7d3e-4c97-b463-8ddc05c07936-0' usage_metadata={'input_tokens': 21, 'output_tokens': 147, 'total_tokens': 168, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audi

In [8]:
from langchain import LLMChain

# 입력 : prompt_template에 전달할 값 -> chain(prompt_template -> llm) -> 출력 : llm의 결과 
chain = LLMChain(
    prompt=prompt_template,
    llm=model
    #output_patser = outputpaser객체
)
result = chain.invoke({'item':'가방', 'count':3})
print(result['text'])

1. **에코백이** - 친환경적인 느낌을 주는 가방에 어울리며, 자연과 조화를 이루는 스타일.
2. **럭셔리 포켓** - 고급스러운 디자인의 가방을 위한 이름으로, 세련된 이미지를 강조.
3. **모던 캐리** - 현대적이고 실용적인 스타일의 가방에 적합한 이름으로, 일상 생활에서 함께하기 좋은 느낌.


In [9]:
#LCEL

chain2 = prompt_template | model
result2 = chain2.invoke({'item':'컴퓨터', 'count':5})
print(result2.content)

물론입니다! 컴퓨터에 어울리는 이름 5개를 제안드립니다:

1. **네오텍 (Neotech)** - 새로운 기술을 상징하는 이름.
2. **코어마스터 (CoreMaster)** - 컴퓨터의 핵심 성능을 강조하는 이름.
3. **디지털스피릿 (DigitalSpirit)** - 디지털 세계의 정신을 의미하는 이름.
4. **인포서지 (InfoSurge)** - 정보의 넘치는 힘을 뜻하는 이름.
5. **큐빅 (Cubic)** - 현대적이고 정돈된 느낌을 주는 이름.

이름이 마음에 드시길 바랍니다!


# LCEL (LangChain Expression Language)
- LCEL은 LangChain의 핵심 기능인 **체인(Chain)을 더욱 효율적으로 구현하기 위해 도입된 **선언적 방식의 체인(chain) 구성 언어**이다.
- `|` 연산자를 이용해 선언적 방법으로 Chain을 만든다.
- [Runnable](https://api.python.langchain.com/en/latest/runnables/langchain_core.runnables.base.Runnable.html) type의 component들이 chain에 포함될 수있다.
    - `|` 연산자를 이용해 Runnable들을 연결한다.
    - chain이 실행되면 각 Runnable의 invoke() 메소드가 실행된다. 그리고 invoke()의 리턴값을 다음 Runnable의 invoke()에 전달해서 실행시킨다.
    - [Runnable 컴포넌트별 입출력 타입](https://python.langchain.com/docs/expression_language/interface)
        - 각 컴포넌트의 input과 output 타입에 맞춰 값이 전달되도록 한다.
- https://python.langchain.com/v0.2/docs/concepts/#langchain-expression-language-lcel

## Runnable
- LangChain의 Runnable은 실행 가능한 작업 단위를 캡슐화한 개념으로, 데이터 흐름의 각 단계를 정의하고 체인(chain) 형태로 연결하여 복잡한 작업을 수행할 수 있게 한다.
- Chain을 구성하는 class들은 Runnable의 하위 클래스로 구현한다.

### 주요 특징
- 작업 단위의 캡슐화:
    - Runnable은 특정 작업(예: 프롬프트 생성, LLM 요청)을 수행하는 독립적인 컴포넌트이다.
    - LangChain의 다양한 컴포넌트(PromptTemplate, LLM, OutputParser 등)들이 Runnable을# Runnable
- LangChain의 Runnable은 실행 가능한 작업 단위를 캡슐화한 개념으로, 데이터 흐름의 각 단계를 정의하고 체인(chain) 형태로 연결하여 복잡한 작업을 수행할 수 있게 한다.
- Chain을 구성하는 class들은 Runnable의 하위 클래스로 구현한다.

### 주요 특징
- 작업 단위의 캡슐화:
    - Runnable은 특정 작업(예: 프롬프트 생성, LLM 요청)을 수행하는 독립적인 컴포넌트이다.
    - LangChain의 다양한 컴포넌트(PromptTemplate, LLM, OutputParser 등)들이 Runnable을 상속받아 구현된다.
- 체인 연결 및 작업 흐름 관리:
    - Runnable은 파이프라인처럼 체인(순차적으로 실행되는 작업들을 연결한 것)을 구성할 수 있으며, `|` 연산자를 사용해 간단히 연결 가능하다.
    - 입력과 출력 형식을 통일해서 컴포넌트를 매끄럽게 연결한다
- 모듈화 및 디버깅 용이성:
    - 각 단계가 명확히 분리되어 디버깅 및 유지보수가 용이하다.
    - 복잡한 작업을 작은 단위로 나누어 관리할 수 있다.
### Runnable의 표준 메소드
- 모든 Runnable이 구현하는 공통 메소드
- `invoke()`: 입력 데이터를 처리하여 결과를 반환.
- `batch()`: 여러 입력 데이터들을 한 번에 처리.
- `stream()`: 스트리밍 방식으로 응답 반환.
- `ainvoke()`: 비동기 호출 지원.

### Runnable의 주요 구현체
- **`RunnablePassThrough`**
    - 입력데이터를 다음 chain으로 그대로 전달하거나, 필요에 따라 추가적인 key-value 쌍을 더해서 전달한다. 
- **`RunnableParallel`**
    - 여러 Runnable을 병렬로 실행하고 결과들을 합쳐서 다음 chain으로 전달한다.`**
- **`RunnableLambda`**
    - 일반 함수나 lambda 함수를 Runnable로 만들 때 사용.

In [11]:
from dotenv import load_dotenv
load_dotenv()

True

In [45]:
### 사용자 정의 Runnable class
from langchain_core.runnables import Runnable

class MyRunnable(Runnable) :
    # invoke는 한 개 파라미터 필수
    def invoke(self, input_data, config=None) :
        if config is not None :
            # chain에서 runnable로 config를 전달할 때 : config={'configurable':{'lang':'en'}}
            if config['configurable']['lang'] == 'en' :
                return f'Explain {input_data} in one sentence'
        return f'{input_data}에 대해서 한 문장으로 설명해줘'

my_runnable = MyRunnable()
my_runnable.invoke('사과')
my_runnable.invoke('아이폰', {'configurable':{'lang':'en'}})

'Explain 아이폰 in one sentence'

In [46]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model='gpt-4o-mini', max_tokens=100)
prompt = my_runnable.invoke('사과')
res = model.invoke(prompt)
print(res.content)

사과는 달콤하고 상큼한 맛을 지닌 과일로, 건강에 유익한 영양소와 항산화 물질이 풍부하여 다양한 요리와 간식으로 즐겨집니다.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
chain = my_runnable | model | StrOutputParser()
result = chain.invoke('seattle', {'lang':'en'})
# chain에서 runnable로 config를 전달할 때 : config={'configurable':{'lang':'en'}}
print(result)


Seattle is a vibrant, tech-driven city in the Pacific Northwest known for its iconic Space Needle, diverse culture, thriving coffee scene, and beautiful natural surroundings, including mountains and water.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 실행 순서 prompt -> model -> outputparser
model = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.1
)
parser = CommaSeparatedListOutputParser()

prompt_template = ChatPromptTemplate(
    [
        ('system', '{format_instruction}. \n목록의 item은 {count}개를 넘지 않도록 해주세요'),
        ('human', '{query}')
    ],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = prompt_template | model | parser

result = chain.invoke({'count':5, 'query':'토스카나에 가볼만한 여행지를 알려줘'})
print(result)


['피렌체', '시에나', '산 지미냐노', '루카', '피사']


In [54]:
# 레시피 요청 - (llm) -> 영어 레시피 - llm -> 한국어로 번역 요청 -> 한국어 레시피
# 1. chain : 레시피 요청 -> 레시피 출력 (영어)
# 2. chain(번역) : 영어 -> 한국어 번역
# 3. 최종 chain : 레시피체인 -> 번역 체인


model = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.1
)

chef_template = ChatPromptTemplate(
    [
        ('system','You are world class international chef. you create easy to follow recipies for any type of cuisine with easy to find ingredients'),
        ('human', 'I want to cook {food} food')
    ]
)

chef_chain = chef_template | model | StrOutputParser()

# c_res = chef_chain.invoke({'food':'steak'})



In [55]:
# 번역 chain
translate_tempate = ChatPromptTemplate(
    [
        ('system', '당신은 번역가입니다. 다음 내역을 한국어로 번역해주세요'),
        ('human', '{query}')
    ]
)

translate_chain = translate_tempate | model | StrOutputParser()

# res = translate_chain.invoke({'query':c_res.content})
# print(res)

In [57]:
final_chain = chef_chain | translate_chain

result = final_chain.invoke({'food':'steak'})
print(result)

물론입니다! 다음은 클래식한 마늘 버터 팬 시어드 스테이크의 간단하고 맛있는 레시피입니다. 이 레시피는 따라하기 쉽고 구하기 쉬운 재료를 사용합니다.

### 마늘 버터 팬 시어드 스테이크

#### 재료:
- 리브아이 또는 등심 스테이크 2개 (약 1인치 두께)
- 소금과 후추 (맛에 따라)
- 올리브 오일 2큰술
- 무염 버터 3큰술
- 다진 마늘 3쪽
- 신선한 허브 (타임이나 로즈마리 등, 선택 사항)
- 레몬 조각 (서빙용)

#### 조리 방법:

1. **스테이크 준비하기:**
   - 스테이크를 냉장고에서 꺼내어 실온에서 약 30분간 두세요. 이렇게 하면 스테이크가 더 고르게 익습니다.
   - 종이 타올로 스테이크의 수분을 제거하고, 양면에 소금과 후추를 넉넉히 뿌려 간을 합니다.

2. **팬 가열하기:**
   - 중-강불로 큰 프라이팬(가급적 주철 팬)에 올리브 오일을 넣고 가열합니다. 오일이 반짝이지만 연기가 나지 않을 정도로 가열합니다.

3. **스테이크 굽기:**
   - 뜨거운 팬에 조심스럽게 스테이크를 올립니다. 팬이 너무 붐비지 않도록 하세요; 필요하다면 한 번에 하나씩 요리합니다.
   - 스테이크를 한쪽 면에서 약 4-5분간 움직이지 않고 구워줍니다. 이렇게 하면 맛있는 크러스트가 생깁니다.

4. **뒤집고 버터 추가하기:**
   - 집게를 사용하여 스테이크를 뒤집습니다. 팬에 버터, 다진 마늘, 허브(사용하는 경우)를 추가합니다.
   - 버터가 녹으면서 숟가락을 사용해 스테이크에 녹은 버터와 마늘을 끼얹어줍니다. 미디엄 레어로 익히려면 추가로 3-4분 더 요리하고, 더 잘 익힌 스테이크를 원하시면 더 오래 익히세요.

5. **익힘 정도 확인하기:**
   - 고기 온도계를 사용하여 내부 온도를 확인합니다:
     - 미디엄 레어: 130°F (54°C)
     - 미디엄: 140°F (60°C)
     - 미디엄 웰: 150°F (65°C)

6. **스테이크 휴지시키기:**
   - 원하는 익힘 정도로 조리

### RunnablePassThrough
1. 입력 값(이전 작업에서 전달한 값)을 다음 체인으로 그대로 넘길 때 사용
2. 입력값에 item(key-value)를 추가해서 다음 체인으로 넘길 때 사용
- 주로 RunnableParallel(동시에 여러 작업을 처리-병렬처리)에 넣어 사용

In [64]:
from langchain_core.runnables import RunnablePassthrough

# 1. 그대로 넘김
r = RunnablePassthrough()
r.invoke({'key':'value'})

# 2. item을 추가
# RunnablePassThrough.assign(key=Callable)
r = RunnablePassthrough.assign(new_key=lambda _input:"new value")
r = RunnablePassthrough.assign(new_key=lambda _input:_input['key'] + ' new value')

# Callable : 파라미터 1개 필수 선언 -> 입력 dictionary
r.invoke({'key':'value'})

{'key': 'value', 'new_key': 'value new value'}

In [74]:

model = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.1
)

chef_template = ChatPromptTemplate(
    [
        ('system','You are world class international chef. you create easy to follow recipies for any type of cuisine with easy to find ingredients'),
        ('human', 'I want to cook {food} food')
    ]
)

chef_chain = chef_template | model | StrOutputParser()


In [78]:
# 번역 chain
translate_tempate = ChatPromptTemplate(
    [
        ('system', '당신은 {language} 번역가입니다. 다음 내역을 {language}로 번역해주세요'),
        ('human', '{query}')
    ]
)

translate_chain = translate_tempate | model | StrOutputParser()

# res = translate_chain.invoke({'language':'불어', 'query':'배불러'})
print(res)

J'ai trop mangé.


In [ ]:
# chain = 작업1 | 작업2 | 작업3

# 한 작업에서 두 개 이상의 Runnable이 실행되어야 할 경우 - RunnableParallel
# chain = 작업1(두개가 병렬로 처리) | => RunnableParallel에 두 개 작업을 묶어줌
# LCEL의 표현식 - {key1:작업A, key2:작업B} | 작업2
# 작업X -> Runnable -> 작업 A가 처리한 결과를 key1, 작업 B가 처이한 결과를 key2에 넣은 뒤 dictionary를 다음 chain으로 넘겨줌

In [ ]:
final_chain = ({'query':chef_chain, 'language':RunnablePassthrough()} | translate_chain)
# chef_chain 입력 : food - final_chain.invoke() 호출 시 food:음식으로 전달
# translate_chain :
#   - query : chef_chain의 output
#   - language : 직접 입력 - final_chain의 입력 중 language key 값 전달
result = final_chain.invoke({'language':'불어', 'food':'steak'})
print(result)

Excellente choix ! Le steak est un plat délicieux et polyvalent qui peut être préparé de différentes manières. Voici une recette simple et facile pour un steak poêlé classique avec du beurre à l'ail. Cette recette sert 2 personnes.

### Steak Poêlé au Beurre à l'Ail

#### Ingrédients :
- 2 steaks ribeye ou sirloin (environ 2,5 cm d'épaisseur)
- Sel et poivre noir (au goût)
- 2 cuillères à soupe d'huile d'olive
- 3 cuillères à soupe de beurre non salé
- 3 gousses d'ail, hachées
- Herbes fraîches (comme le thym ou le romarin) pour la garniture (facultatif)

#### Instructions :

1. **Préparer les Steaks :**
   - Sortez les steaks du réfrigérateur et laissez-les reposer à température ambiante pendant environ 30 minutes. Cela les aide à cuire plus uniformément.
   - Séchez les steaks avec du papier absorbant. Assaisonnez généreusement les deux côtés avec du sel et du poivre noir.

2. **Chauffer la Poêle :**
   - Dans une grande poêle (de préférence en fonte) à feu moyen-vif, ajoutez l'huile

## 사용자 함수를 chain으로 정의
- 임의의 함수를 Runnable로 정의 할 수있다.
    - LangChain 에서 제공하지 않는 기능을 Chain으로 만들 때 유용한다.
- LangChain에서는 Runnable로 사용되는 사용자 정의 함수를 **Runnable Lambda** 라고 한다.
- 함수를 Runnable 로 명시하는데는 다음 두가지 방법이 있다.
1. `RunnableLambda` 이용
   - `RunnableLambda(함수)`
3. `@chain` 데코레이터 이용
   - ```python
     @chain
     def func():
         ...
    ```
### Runnable 로 정의 하는 함수 정의
- 이전 Chain의 출력을 입력 받는 파라미터를 한개 선언한다.
- 만약 함수가 여러개의 인자를 받는 경우 단일 입력을 받아들이고 이를 여러 인수로 풀어내는 래퍼 함수를 작성하여 Runnable로 만든다.
```python
def plus(num1, num2):
    ...

def wrapper_plus(nums:dict|list):
    return plus(nums['num1'], nums['num2'])
```
- Chain의 실행결과를 return 한다.

In [85]:
from langchain_core.runnables import RunnableLambda
# RunnableLambda(함수) : func-필수 parameter:1 (invoke를 통해 전달된 인수값)
r_lambda = RunnableLambda(lambda _input : f'{_input}에 대해서 설명해줘')
r_lambda.invoke('Python')

chain = r_lambda | model
res = chain.invoke('python')
print(res.content)

Python은 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 개발된 고급 프로그래밍 언어입니다. Python은 간결하고 읽기 쉬운 문법을 가지고 있어 초보자부터 전문가까지 널리 사용되고 있습니다. 다음은 Python의 주요 특징과 장점입니다.

### 주요 특징

1. **간결한 문법**: Python은 코드가 명확하고 간결하여 읽기 쉽습니다. 이는 개발자가 코드를 작성하고 유지보수하는 데 도움을 줍니다.

2. **다양한 용도**: Python은 웹 개발, 데이터 분석, 인공지능, 머신러닝, 자동화 스크립트, 게임 개발 등 다양한 분야에서 사용됩니다.

3. **풍부한 라이브러리**: Python은 다양한 표준 라이브러리와 서드파티 라이브러리를 제공합니다. 예를 들어, NumPy, Pandas, Matplotlib, TensorFlow, Django 등이 있습니다.

4. **인터프리터 언어**: Python은 인터프리터 언어로, 코드를 한 줄씩 실행할 수 있어 디버깅이 용이합니다.

5. **객체 지향 프로그래밍**: Python은 객체 지향 프로그래밍(OOP)을 지원하여 코드의 재사용성과 모듈화를 촉진합니다.

6. **크로스 플랫폼**: Python은 Windows, macOS, Linux 등 다양한 운영 체제에서 실행될 수 있습니다.

### 장점

- **학습 용이성**: Python은 초보자에게 적합한 언어로, 프로그래밍의 기본 개념을 배우기에 좋습니다.
- **커뮤니티 지원**: Python은 활발한 커뮤니티가 있어, 문제 해결이나 정보 공유가 용이합니다.
- **다양한 프레임워크**: 웹 개발을 위한 Django, Flask와 같은 프레임워크가 있어 빠른 개발이 가능합니다.
- **데이터 과학 및 AI**: 데이터 분석과 머신러닝을 위한 강력한 라이브러리들이 있어 데이터 과학 분야에서 인기가 높습니다.

### 결론

Python은 그 유연성과 강력한 기능 덕분에 많은 개발자와 기업에서 선호하는 언어입니다. 초보자부터 전

In [91]:
from langchain_core.runnables import chain
# @chain - chain을 구성 할 때 사용 (RunnableLambda - 개별 작업 단위 구현)

@chain  # 붙여주면 runnable 객체됨
def custom_chain(topic:str) :
    story_prompt = PromptTemplate(
        template='{topic}에 대한 재미있는 얘기를 만들어줘 이야기는 30문장 이내로 만들어줘'
    )
    sotry_model = ChatOpenAI(model='gpt-4o', temperature=1)
    story_chain = story_prompt | sotry_model | StrOutputParser()
    story_result = story_chain.invoke({'topic':topic})

    summary_prompt = PromptTemplate(
        template='다음 내용을 2문장으로 요약해줘\n[요약할 내용]\n{content}'
    )
    summary_model = ChatOpenAI(model='gpt-4o-mini')
    summary_chain = summary_prompt | summary_model | StrOutputParser()
    summary_result = summary_chain.invoke({'content':story_result})

    return {'story':story_result, 'summary':summary_result}

res = custom_chain.invoke('커피')
res

{'story': "하루는 작고 조용한 마을의 한 카페에서 시작되었습니다. 이 카페는 특이하게도 긴 줄을 서지 않고는 들어갈 수 없는 곳이었어요. 비결은 바로 '마법의 커피콩'이었습니다. 이 커피콩은 주인이 여행 중 한 신비한 산악 마을에서 발견한 것이었죠.\n\n마을 사람들은 이 커피콩을 마법의 커피콩이라고 불렀고, 한 번 맛보면 누구든지 그것을 잊을 수 없었습니다. 그래서 카페 주인의 이름은 점점 멀리 퍼져나갔고, 사람들은 이 특별한 커피를 맛보기 위해 먼 길도 마다하지 않았어요.\n\n하지만 이 커피콩은 단순히 맛만 좋은 게 아니었어요. 마을의 전설에 따르면, 이 커피를 마시면 마음 깊은 곳의 소원이 이루어진다고 했죠. 어느 날, 마을의 한 젊은 예술가가 그 전설을 실험하고 싶어 카페를 찾았습니다. 그 예술가는 오랫동안 세상에 알려지지 않은 채 그림을 그리고 있었고, 그의 소원은 자신의 작품이 언젠간 빛을 보는 것이었습니다.\n\n예술가는 커피를 한 모금 마신 뒤, 그의 마음 속 깊은 곳에서 상상했던 모든 것들이 실제로 보이기 시작했어요. 그의 손은 마치 구름을 그리는 듯 자연스럽게 움직였고, 그 결과 눈부시게 아름다운 작품이 완성되었습니다. 얼마 지나지 않아 그 작품은 대중의 주목을 받게 되었고, 예술가는 큰 성공을 거두었습니다.\n\n시간이 흐르면서 더 많은 사람들이 자신의 소원을 이루기 위해 카페를 찾았습니다. 누군가는 사랑을 찾고, 누군가는 잃어버린 가족을 찾고, 또 다른 누군가는 단순한 행복을 찾아왔지만, 모두가 그 특별한 커피의 힘을 믿게 되었어요.\n\n하지만 이 모든 일이 벌어지는 동안, 카페 주인은 그저 조용히 미소 지으며 커피를 내렸습니다. 그는 알고 있었죠. 진정한 마법은 바로 커피를 마시는 사람들의 믿음과 희망에 있다는 것을. 그리고 그것이야말로 커피가 가지는 최고의 맛이라는 것을 말입니다. \n\n그렇게 이 작은 마을은 커피의 작은 마법 덕분에 희망과 이야기로 가득 찬 곳이 되었습니다. 그리고 카페 주인은 여전히 그 자리에서 커피를

# Cache

- 응답 결과를 저장해서 같은 질문이 들어오면 LLM에 요청하지 않고 저장된 결과를 보여주도록 한다.
    - 처리속도와 비용을 절감할 수 있다.
    - 특히 chatbot같이 비슷한 질문을 하는 경우 유용하다.
- 저장 방식은 `메모리`, `sqlite` 등 다양한 방식을 지원한다.
    - https://python.langchain.com/docs/integrations/llms/llm_caching
```python
set_llm_cache(Cache객체)
```

In [99]:
from langchain.cache import InMemoryCache, SQLiteCache
from langchain.globals import set_llm_cache

# model 호출 전에 set_llm_cache를 실행해서 cache 사용을 선언하면 됨
# set_llm_cache(InMemoryCache())
set_llm_cache(SQLiteCache('cache.sqlite')) # file db로 만듦

res = final_chain.invoke({'food':'pasta', 'language':'한국어'})

In [95]:
res2 = final_chain.invoke({'food':'pasta', 'language':'한국어'})